In [7]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
import pandas as pd
import geopandas as gpd # Importado como gpd por convención

# ---------------------------------------------------------
# 1. CONFIGURACIÓN DE CONEXIÓN Y PROTOCOLOS
# ---------------------------------------------------------
host = "192.168.2.195"
port = 5432
dbname = "geonode_local_data"
user = "guest3"
password = "guest3_2026_merlin"

# Define aquí tu identificador/prefijo según la regla de tu equipo
# Ejemplo: si eres 'guest1', tus tablas se llamarán 'guest1_boundaries_comunal', etc.
mi_prefijo = "guest3" 
esquema_destino = "work"

url = URL.create(
    drivername="postgresql+psycopg2",
    username=user,
    password=password,
    host=host,
    port=port,
    database=dbname,
)

# pool_pre_ping=True ayuda a evitar errores de desconexión silenciosa
engine = create_engine(url, pool_pre_ping=True)

# ---------------------------------------------------------
# 2. SUBIR GEOCAPAS (Polígonos espaciales)
# ---------------------------------------------------------
print("--- Iniciando subida de Geocapas (.gpkg) ---")

capas = {
    "../data/rec_2024_2025/results/capas_comunales/wp2_output_demanda_electrica_comunal.gpkg": f"{mi_prefijo}_boundaries_comunal_elec_2024_2025",
    "../data/rec_2024_2025/results/capas_regionales/wp2_output_demanda_electrica_regional.gpkg": f"{mi_prefijo}_boundaries_regional_elec_2024_2025"
}

for archivo, nombre_tabla in capas.items():
    print(f"Leyendo {archivo}...")
    gdf = gpd.read_file(archivo)
    
    # Opcional pero recomendado: Asegurar que los nombres de columnas estén en minúsculas
    gdf.columns = [col.lower() for col in gdf.columns]
    
    print(f"Inyectando en la base de datos: {esquema_destino}.{nombre_tabla} ...")
    # Usamos to_postgis para subir la geometría de forma nativa a PostGIS
    gdf.to_postgis(
        name=nombre_tabla,
        con=engine,
        schema=esquema_destino,
        if_exists="replace", # Usa 'replace' para sobrescribir si te equivocas, o 'append' para añadir
        index=False          # Evita subir el índice de Pandas como una columna extra
    )
    print("¡Subida exitosa!\n")

# ---------------------------------------------------------
# 3. SUBIR SERIES DE TIEMPO (Datos tabulares)
# ---------------------------------------------------------
print("--- Iniciando subida de Series de Tiempo (.parquet) ---")

series_tiempo = {
    "../data/rec_2024_2025/results/capas_comunales/wp2_output_demanda_electrica_comunal_ts.parquet": f"{mi_prefijo}_ts_comunal_elec_2024_2025",
    "../data/rec_2024_2025/results/capas_regionales/wp2_output_demanda_electrica_regional_ts.parquet": f"{mi_prefijo}_ts_regional_elec_2024_2025"
}

for archivo, nombre_tabla in series_tiempo.items():
    print(f"Leyendo {archivo}...")
    df = pd.read_parquet(archivo)
    
    # Asegurar nombres de columnas en minúsculas para PostgreSQL
    df.columns = [col.lower() for col in df.columns]
    
    print(f"Inyectando en la base de datos: {esquema_destino}.{nombre_tabla} ...")
    # Usamos to_sql para bases tabulares clásicas
    df.to_sql(
        name=nombre_tabla,
        con=engine,
        schema=esquema_destino,
        if_exists="replace", 
        index=False
    )
    print("¡Subida exitosa!\n")

print("=== PROCESO COMPLETADO ===")

--- Iniciando subida de Geocapas (.gpkg) ---
Leyendo ../data/rec_2024_2025/results/capas_comunales/wp2_output_demanda_electrica_comunal.gpkg...
Inyectando en la base de datos: work.guest3_boundaries_comunal_elec_2024_2025 ...
¡Subida exitosa!

Leyendo ../data/rec_2024_2025/results/capas_regionales/wp2_output_demanda_electrica_regional.gpkg...
Inyectando en la base de datos: work.guest3_boundaries_regional_elec_2024_2025 ...
¡Subida exitosa!

--- Iniciando subida de Series de Tiempo (.parquet) ---
Leyendo ../data/rec_2024_2025/results/capas_comunales/wp2_output_demanda_electrica_comunal_ts.parquet...
Inyectando en la base de datos: work.guest3_ts_comunal_elec_2024_2025 ...
¡Subida exitosa!

Leyendo ../data/rec_2024_2025/results/capas_regionales/wp2_output_demanda_electrica_regional_ts.parquet...
Inyectando en la base de datos: work.guest3_ts_regional_elec_2024_2025 ...
¡Subida exitosa!

=== PROCESO COMPLETADO ===
